# Stress Tensor Inversion

This notebook demonstrates **`swe.stress`** for stress tensor inversion from
focal mechanism data. We cover:

1. Generating synthetic focal mechanisms
2. Clustering focal mechanisms by orientation
3. Stress tensor inversion
4. Visualization (stereonets, principal stress directions)

**Dataset**: Synthetic focal mechanisms drawn from a known stress state.

In [ ]:
from __future__ import annotations

import logging

import matplotlib.pyplot as plt
import numpy as np

import swe.stress

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

%matplotlib inline

## Configuration

In [ ]:
N_EVENTS = 50  # number of synthetic focal mechanisms
SEED = 42

# True stress field (strike-slip regime)
# S1 (max compressive) azimuth ~30°, plunge ~0°
# S3 (min compressive) azimuth ~120°, plunge ~0°
# S2 vertical

## 1. Generate Synthetic Focal Mechanisms

We create focal mechanisms (strike, dip, rake) consistent with a
strike-slip stress regime, adding Gaussian noise to simulate real
measurement uncertainty.

In [ ]:
rng = np.random.default_rng(SEED)

# Base strike-slip mechanisms:
# Nodal plane 1: strike=30°, dip=90°, rake=0° (left-lateral)
# Nodal plane 2: strike=120°, dip=90°, rake=180° (right-lateral)
strikes = np.concatenate([
    rng.normal(30, 10, N_EVENTS // 2),
    rng.normal(120, 10, N_EVENTS // 2),
]) % 360

dips = np.clip(rng.normal(85, 5, N_EVENTS), 30, 90)
rakes = np.concatenate([
    rng.normal(0, 15, N_EVENTS // 2),
    rng.normal(180, 15, N_EVENTS // 2),
])
rakes = ((rakes + 180) % 360) - 180  # wrap to [-180, 180]

focal_data = np.column_stack([strikes, dips, rakes])
print(f"Generated {len(focal_data)} focal mechanisms")
print(f"Strike range: {strikes.min():.1f}° - {strikes.max():.1f}°")
print(f"Dip range: {dips.min():.1f}° - {dips.max():.1f}°")
print(f"Rake range: {rakes.min():.1f}° - {rakes.max():.1f}°")

## 2. Cluster Focal Mechanisms

Group similar focal mechanisms to identify distinct faulting populations.

In [ ]:
cluster_result = swe.stress.cluster_focal_mechanisms(
    focal_data, n_clusters=2,
)

print(f"Number of clusters: {cluster_result.n_clusters}")
for i in range(cluster_result.n_clusters):
    mask = cluster_result.labels == i
    n = mask.sum()
    mean_strike = np.mean(focal_data[mask, 0])
    mean_dip = np.mean(focal_data[mask, 1])
    mean_rake = np.mean(focal_data[mask, 2])
    print(f"  Cluster {i}: {n} events, mean (strike={mean_strike:.1f}°, "
          f"dip={mean_dip:.1f}°, rake={mean_rake:.1f}°)")

## 3. Split Focal Data

Prepare the data format needed by the stress inversion.

In [ ]:
split_data = swe.stress.split_focal_data(focal_data)
print(f"Split data shape: {split_data.shape}")

## 4. Stress Tensor Inversion

Invert for the principal stress directions and stress ratio (R-value)
using the Michael (1984) method.

In [ ]:
stress_result = swe.stress.stress_inversion(focal_data)

print(f"Stress ratio (R): {stress_result.r_value:.3f}")
print(f"Misfit: {stress_result.misfit:.4f}")
print(f"\nPrincipal stress directions:")
print(f"  S1 (max): azimuth={stress_result.s1_azimuth:.1f}°, plunge={stress_result.s1_plunge:.1f}°")
print(f"  S2 (int): azimuth={stress_result.s2_azimuth:.1f}°, plunge={stress_result.s2_plunge:.1f}°")
print(f"  S3 (min): azimuth={stress_result.s3_azimuth:.1f}°, plunge={stress_result.s3_plunge:.1f}°")

## 5. Visualization

In [ ]:
# Strike-dip-rake distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(focal_data[:, 0], bins=30, edgecolor="k", alpha=0.7)
axes[0].set_xlabel("Strike (°)")
axes[0].set_title("Strike Distribution")

axes[1].hist(focal_data[:, 1], bins=20, edgecolor="k", alpha=0.7, color="orange")
axes[1].set_xlabel("Dip (°)")
axes[1].set_title("Dip Distribution")

axes[2].hist(focal_data[:, 2], bins=30, edgecolor="k", alpha=0.7, color="green")
axes[2].set_xlabel("Rake (°)")
axes[2].set_title("Rake Distribution")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cluster membership coloured by group
fig, ax = plt.subplots(figsize=(8, 6))
for i in range(cluster_result.n_clusters):
    mask = cluster_result.labels == i
    ax.scatter(
        focal_data[mask, 0], focal_data[mask, 2],
        label=f"Cluster {i}", alpha=0.7, s=40,
    )
ax.set_xlabel("Strike (°)")
ax.set_ylabel("Rake (°)")
ax.set_title("Focal Mechanisms by Cluster")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `cluster_focal_mechanisms()` | Group similar focal mechanisms |
| `split_focal_data()` | Prepare data for inversion |
| `stress_inversion()` | Invert for stress tensor (S1, S2, S3, R) |

The inverted stress directions should recover the input strike-slip regime:
- S1 ~N30°E (horizontal, maximum compression)
- S3 ~N120°E (horizontal, minimum compression)
- S2 ~vertical